# Phase 1 — UniMatch Semi-Supervised Training (Dev)

# Segmentation + Classification | ResNet-50 + UNETR

In [ ]:
import torch
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()


In [ ]:
import sys
import os
from pathlib import Path
import json
import logging
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import SGD
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
import time
import cv2
from collections import defaultdict, Counter

from dataset.fetus_v2 import FETUSSemiDataset
from model.echocare_resnet import Echocare_ResNet
from src.utils import (
    DiceLoss, AverageMeter, nsd_binary,
    masked_bce_with_logits, masked_mse,
    build_allowed_mat, apply_view_mask_logits,
    compute_pos_weight_from_loader,
    masked_metrics_with_threshold_search,
    build_same_view_perm,
)

from dataset.transform_v2 import load_view_stats
load_view_stats("preprocessing_stats.json")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
def setup_logger(log_dir: str, name: str = "UniMatch") -> logging.Logger:
    os.makedirs(log_dir, exist_ok=True)
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    if logger.handlers:
        logger.handlers = []
    fmt = logging.Formatter("[%(asctime)s] %(message)s", datefmt="%H:%M:%S")
    fh = logging.FileHandler(os.path.join(log_dir, "train.log"))
    fh.setFormatter(fmt)
    sh = logging.StreamHandler(sys.stdout)
    fh.setFormatter(fmt)
    sh.setFormatter(fmt)
    logger.addHandler(fh)
    logger.addHandler(sh)
    return logger

print("Logger function defined")


In [ ]:
def masked_focal_bce_with_logits(
    logits: torch.Tensor,
    targets: torch.Tensor,
    mask: torch.Tensor,
    pos_weight: torch.Tensor = None,
    gamma: float = 2.0,
    alpha: float | None = None,
    eps: float = 1e-8,
):
    """
    Focal BCE with logits + optional pos_weight (per class) + mask (per-sample/per-class).
    logits:  [B, C]
    targets: [B, C] float (0/1)
    mask:    [B, C] float/bool (1=use, 0=ignore)
    pos_weight: [C] (like BCEWithLogits pos_weight)
    """
    mask = mask.float()

    bce = F.binary_cross_entropy_with_logits(
        logits, targets, reduction="none", pos_weight=pos_weight
    )  # [B, C]

    p = torch.sigmoid(logits)
    pt = p * targets + (1.0 - p) * (1.0 - targets)  # prob of the true class
    focal = (1.0 - pt).pow(gamma)

    if alpha is not None:
        alpha_t = alpha * targets + (1.0 - alpha) * (1.0 - targets)
        focal = focal * alpha_t

    loss = bce * focal
    loss = loss * mask

    return loss.sum() / (mask.sum() + eps)


In [ ]:
_json_dir = Path.cwd()

class Config:
    epochs     = 50
    batch_size = 16
    base_lr    = 0.001
    version    = "PHASE1_DEV"
    amp_dtype  = "fp16"

    conf_thresh     = 0.9
    pseudo_tau_pos  = 0.8
    pseudo_tau_neg  = 0.8

    seg_num_classes  = 15
    cls_num_classes  = 7
    view_num_classes = 4
    resize_target    = 256

    train_labeled_json   = str(_json_dir / "train_labeled.json")
    train_unlabeled_json = str(_json_dir / "train_unlabeled.json")
    valid_json           = str(_json_dir / "valid.json")

    checkpoint_dir = os.path.join("checkpoints", version)
    log_dir        = os.path.join("logs",        version)
    result_dir     = os.path.join("results",     version)
    warmup_epochs = 0
    num_workers = 0
    amp         = True

cfg = Config()
print(f"Version:  {cfg.version}")
print(f"Epochs:   {cfg.epochs} | BS: {cfg.batch_size} | LR: {cfg.base_lr}")
print(f"AMP:      {cfg.amp} ({cfg.amp_dtype})")
print(f"Checkpoints -> {cfg.checkpoint_dir}")

In [ ]:
SEG_ALLOWED = {
    0: [0, 1, 2, 3, 4, 5, 6, 7],    # 4CH
    1: [0, 1, 2, 4, 8],              # LVOT
    2: [0, 6, 8, 9, 10, 11, 12],     # RVOT
    3: [0, 9, 12, 13, 14],           # 3VT
}

CLS_ALLOWED = {
    0: [0, 1],
    1: [0, 2, 3],
    2: [4, 5],
    3: [2, 5, 6],
}

LOSS_WEIGHTS = {
    "x_seg":         1.0,
    "x_cls":         1.0,
    "x_fp_cls":      1.0,
    "u_s1_seg":      0.25,
    "u_s2_seg":      0.25,
    "u_w_fp_seg":    0.5,
    "u_s1_mix_seg":  0.25,
    "u_s2_mix_seg":  0.25,
    "u_pseudo_cls":  1.0,
    "u_w_mix_cls":   0.25,
    "u_w_mix_fp_cls": 0.5,
}

print("SEG_ALLOWED:", SEG_ALLOWED)
print("CLS_ALLOWED:", CLS_ALLOWED)


In [ ]:
def build_seg_allowed_mat(device, seg_allowed, num_views, num_classes):
    mat = torch.zeros((num_views, num_classes), dtype=torch.bool, device=device)
    for v in range(num_views):
        if v not in seg_allowed:
            raise ValueError(f"seg_allowed missing view {v}")
        mat[v, seg_allowed[v]] = True
    return mat

allowed_seg_mat = build_seg_allowed_mat(
    device, SEG_ALLOWED, cfg.view_num_classes, cfg.seg_num_classes
)
allowed_cls_mat = build_allowed_mat(
    device, CLS_ALLOWED,
    num_views=cfg.view_num_classes,
    num_classes=cfg.cls_num_classes
)

print(f"Seg allowed matrix:  {allowed_seg_mat.shape}")   # (4, 15)
print(f"Cls allowed matrix:  {allowed_cls_mat.shape}")   # (4,  7)


In [ ]:



def poly_lr(base_lr, iters, total_iters, power=0.9):
    return base_lr * (1 - iters / total_iters) ** power

print("poly_lr defined")


In [ ]:
dataset_train_u = FETUSSemiDataset(
    cfg.train_unlabeled_json,
    mode="train_u",
    size=cfg.resize_target
)

dataset_train_l = FETUSSemiDataset(
    cfg.train_labeled_json,
    mode="train_l",
    size=cfg.resize_target,
    n_sample=len(dataset_train_u.case_list)

)

dataset_valid = FETUSSemiDataset(
    cfg.valid_json,
    mode="valid"
)

print(f"Train labeled   : {len(dataset_train_l)}")
print(f"Train unlabeled : {len(dataset_train_u)}")
print(f"Valid           : {len(dataset_valid)}")


In [ ]:
train_loader_l = DataLoader(
    dataset_train_l, batch_size=cfg.batch_size,
    shuffle=True, num_workers=cfg.num_workers,
    drop_last=True, pin_memory=True
)
train_loader_u = DataLoader(
    dataset_train_u, batch_size=cfg.batch_size,
    shuffle=True, num_workers=cfg.num_workers,
    drop_last=True, pin_memory=True
)
train_loader_u_mix = DataLoader(
    dataset_train_u, batch_size=cfg.batch_size,
    shuffle=True, num_workers=cfg.num_workers,
    drop_last=True, pin_memory=True
)
valid_loader = DataLoader(
    dataset_valid, batch_size=1,
    shuffle=False, num_workers=cfg.num_workers,
    pin_memory=True
)

print(f"Labeled   batches : {len(train_loader_l)}")
print(f"Unlabeled batches : {len(train_loader_u)}")
print(f"Valid     batches : {len(valid_loader)}")


In [ ]:
model = Echocare_ResNet(
    seg_class_num=cfg.seg_num_classes,
    cls_class_num=cfg.cls_num_classes,
    pretrained=True,
).to(device)

total     = sum(p.numel() for p in model.parameters()) / 1e6
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"Echocare_ResNet: {total:.2f}M total | {trainable:.2f}M trainable")

In [ ]:
backbone_layers = ("seg_net.pre_pool", "seg_net.post_pool",
                   "seg_net.layer1", "seg_net.layer2",
                   "seg_net.layer3", "seg_net.layer4")
backbone_params, head_params = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if any(name.startswith(k) for k in backbone_layers):
        backbone_params.append(param)
    else:
        head_params.append(param)

base_backbone_lr, base_head_lr = 1e-4, 1e-3
optimizer = torch.optim.AdamW(
    [{"params": backbone_params, "lr": base_backbone_lr},
     {"params": head_params,     "lr": base_head_lr}],
    weight_decay=0.01
)
base_lrs = [base_backbone_lr, base_head_lr]
print(f"AdamW | backbone: {sum(p.numel() for p in backbone_params)/1e6:.2f}M (lr={base_backbone_lr})"
      f" | decoder+heads: {sum(p.numel() for p in head_params)/1e6:.2f}M (lr={base_head_lr})")

criterion_seg_ce   = nn.CrossEntropyLoss()
criterion_seg_dice = DiceLoss(n_classes=cfg.seg_num_classes)

from src.utils import BoundaryLoss, compute_boundary_gt
criterion_boundary = BoundaryLoss(n_classes=cfg.seg_num_classes)

pos_weight = compute_pos_weight_from_loader(train_loader_l, allowed_cls_mat, cfg.cls_num_classes, device)
print(f"pos_weight: {pos_weight}")

use_amp = cfg.amp and torch.cuda.is_available()
amp_dtype = torch.float16 if cfg.amp_dtype == "fp16" else torch.bfloat16
scaler = torch.amp.GradScaler("cuda") if (use_amp and amp_dtype == torch.float16) else None
print(f"AMP: {use_amp} | dtype: {cfg.amp_dtype}")

In [ ]:
@torch.no_grad()
def teacher_pseudo(model, image_u_w_mix, view_u_mix,
                   allowed_seg_mat, allowed_cls_mat,
                   use_hard_view_mask=True, tau_pos=0.8, tau_neg=0.8):
    model.eval()
    pred_u_w_mix, cls_u_w_mix = model(image_u_w_mix)
    p_t = torch.sigmoid(cls_u_w_mix)

    conf_mask  = ((p_t >= tau_pos) | (p_t <= 1 - tau_neg)).float()
    mask_u_cls = conf_mask * allowed_cls_mat[view_u_mix]

    if use_hard_view_mask:
        pred_u_w_mix = apply_view_mask_logits(pred_u_w_mix, view_u_mix, allowed_seg_mat)

    conf_u_w_mix = pred_u_w_mix.float().softmax(dim=1).max(dim=1)[0]
    mask_u_w_mix = pred_u_w_mix.argmax(dim=1)

    return p_t, mask_u_cls, conf_u_w_mix, mask_u_w_mix

print("teacher_pseudo defined")


In [ ]:
SEG_CLASS_NAMES = ["BG","LA","LV","RA","RV","WH","DescAo","Thorax","AscAo","MainPA","LPA","RPA","SVC","AoArch","Trachea"]
CLS_CLASS_NAMES = ["VSD","AV_sten","AoHypo","AoV_sten","DORV","PV_sten","RAA"]
VIEW_NAMES      = ["4CH","LVOT","RVOT","3VT"]

In [ ]:
def train_one_epoch(model, optimizer, scaler,
                    train_loader_l, train_loader_u, train_loader_u_mix,
                    allowed_seg_mat, allowed_cls_mat, pos_weight,
                    criterion_seg_ce, criterion_seg_dice,
                    loss_w, base_lrs,
                    epoch, total_iters, cfg, writer, global_step, logger):
    model.train()

    meters = {k: AverageMeter() for k in [
        "loss", "x_seg", "x_cls", "x_fp_cls",
        "u_s1_seg", "u_s2_seg", "u_w_fp_seg",
        "u_s1_mix_seg", "u_s2_mix_seg",
        "u_pseudo_cls", "u_w_mix_cls", "u_w_mix_fp_cls",
        "seg_mask_ratio", "cls_mask_ratio",
    ]}

    diag_seg_pixels_total    = torch.zeros(cfg.seg_num_classes,  device=device)
    diag_seg_pixels_accepted = torch.zeros(cfg.seg_num_classes,  device=device)
    diag_seg_conf_sum        = torch.zeros(cfg.seg_num_classes,  device=device)
    diag_cls_accepted_per_view = torch.zeros(cfg.view_num_classes, device=device)
    diag_cls_total_per_view    = torch.zeros(cfg.view_num_classes, device=device)
    diag_cls_prob_sum          = torch.zeros(cfg.cls_num_classes,  device=device)
    diag_cls_prob_count        = torch.zeros(cfg.cls_num_classes,  device=device)
    diag_n_unsup_iters = 0

    is_warmup = epoch < cfg.warmup_epochs

    use_hard_view_mask = True
    use_amp   = cfg.amp and (device.type == "cuda")
    amp_dtype = torch.float16 if cfg.amp_dtype == "fp16" else torch.bfloat16

    iters_per_epoch = len(train_loader_u)

    for i, (
        (image_x,       view_x,     mask_x,         class_label_x),
        (image_u_w,     view_u,     image_u_s1,     image_u_s2,    cutmix_box1, cutmix_box2),
        (image_u_w_mix, view_u_mix, image_u_s1_mix, image_u_s2_mix, _, _),
    ) in enumerate(zip(train_loader_l, train_loader_u, train_loader_u_mix)):

        image_x       = image_x.to(device)
        view_x        = view_x.to(device).long().view(-1)
        mask_x        = mask_x.to(device)
        class_label_x = class_label_x.to(device)

        image_u_w   = image_u_w.to(device)
        view_u      = view_u.to(device).long().view(-1)
        image_u_s1  = image_u_s1.to(device)
        image_u_s2  = image_u_s2.to(device)
        cutmix_box1 = cutmix_box1.to(device)
        cutmix_box2 = cutmix_box2.to(device)

        image_u_w_mix   = image_u_w_mix.to(device)
        view_u_mix      = view_u_mix.to(device).long().view(-1)
        image_u_s1_mix  = image_u_s1_mix.to(device)
        image_u_s2_mix  = image_u_s2_mix.to(device)

        perm           = build_same_view_perm(view_u, view_u_mix)
        image_u_w_mix  = image_u_w_mix[perm]
        image_u_s1_mix = image_u_s1_mix[perm]
        image_u_s2_mix = image_u_s2_mix[perm]
        view_u_mix     = view_u_mix[perm]

        mask_x_allowed = allowed_cls_mat[view_x]
        num_l = image_x.shape[0]
        num_u = image_u_w.shape[0]


        model.train()
        with torch.cuda.amp.autocast(enabled=use_amp, dtype=amp_dtype):
            if is_warmup:
                (preds_l, preds_l_fp), (preds_class_l, preds_class_l_fp) = model(image_x, True)
                pred_x          = preds_l
                pred_x_class    = preds_class_l
                pred_x_class_fp = preds_class_l_fp
            else:
                (preds, preds_fp), (preds_class, preds_class_fp) = model(
                    torch.cat((image_x, image_u_w)), True
                )
                pred_x,          pred_u_w    = preds.split([num_l, num_u])
                _,               pred_u_w_fp = preds_fp.split([num_l, num_u])
                pred_x_class,    _           = preds_class.split([num_l, num_u])
                pred_x_class_fp, _           = preds_class_fp.split([num_l, num_u])

            if use_hard_view_mask:
                pred_x = apply_view_mask_logits(pred_x, view_x, allowed_seg_mat)

            (pred_x_main,
             pred_x_aux4,
             pred_x_aux3,
             pred_x_aux2,
             pred_x_aux1,
             pred_x_boundary), pred_x_class_ds = model(image_x, need_fp=False, return_aux=True)



            pred_x = apply_view_mask_logits(pred_x_main, view_x, allowed_seg_mat)

            # Loss principal (salida 256×256)
            loss_main = (
                criterion_seg_ce(pred_x, mask_x) +
                criterion_seg_dice(pred_x.softmax(dim=1), mask_x.unsqueeze(1).float())
            ) / 2.0

            # BoundaryLoss sobre salida principal
            loss_boundary = criterion_boundary(pred_x.softmax(dim=1), mask_x)



            target_size = mask_x.shape[-2:]
            def aux_loss(pred_aux):
                pred_up = F.interpolate(pred_aux, size=target_size,
                                        mode="bilinear", align_corners=False)
                return (
                    criterion_seg_ce(pred_up, mask_x) +
                    criterion_seg_dice(pred_up.softmax(dim=1), mask_x.unsqueeze(1).float())
                ) / 2.0

            loss_ds = (
                DS_WEIGHTS["aux4"] * aux_loss(pred_x_aux4) +
                DS_WEIGHTS["aux3"] * aux_loss(pred_x_aux3) +
                DS_WEIGHTS["aux2"] * aux_loss(pred_x_aux2) +
                DS_WEIGHTS["aux1"] * aux_loss(pred_x_aux1)
            )

            # Boundary Head Loss
            # Morphological boundary GT from mask
            boundary_gt = compute_boundary_gt(mask_x)   # (B, 1, H, W)

            boundary_pred_up = F.interpolate(
                pred_x_boundary, size=mask_x.shape[-2:],
                mode="bilinear", align_corners=False
            )   # (B, 1, H, W)
            loss_boundary_head = criterion_boundary_bce(boundary_pred_up, boundary_gt)

            # Loss supervisado total
            loss_x_seg = (
                DS_WEIGHTS["main"] * loss_main
                + loss_ds
                + BOUNDARY_WEIGHT * loss_boundary
                + BOUNDARY_HEAD_WEIGHT * loss_boundary_head
            )

            # Pred_x para el resto del loop (semisup usa solo la salida principal)
            pred_x = pred_x_main

            loss_x_cls    = masked_focal_bce_with_logits(pred_x_class_ds,    class_label_x.float(), mask_x_allowed, pos_weight, gamma=2.0)
            loss_x_fp_cls = torch.tensor(0.0, device=device)


            loss = (
                loss_w["x_seg"]   * loss_x_seg
                + loss_w["x_cls"] * loss_x_cls
            )

        #  SEMI-SUPERVISADO
        loss_u_s1_seg = loss_u_s2_seg = loss_u_w_fp_seg = 0.0
        loss_u_s1_mix_seg = loss_u_s2_mix_seg = 0.0
        loss_u_pseudo_cls = loss_u_w_mix_cls = loss_u_w_mix_fp_cls = 0.0
        seg_mask_ratio = cls_mask_ratio = 0.0

        if not is_warmup:
            with torch.cuda.amp.autocast(enabled=use_amp, dtype=amp_dtype):

                with torch.no_grad():
                    p_t, mask_u_cls, conf_u_w_mix, mask_u_w_mix = teacher_pseudo(
                        model, image_u_w_mix, view_u_mix,
                        allowed_seg_mat, allowed_cls_mat,
                        use_hard_view_mask=use_hard_view_mask,
                        tau_pos=cfg.pseudo_tau_pos, tau_neg=cfg.pseudo_tau_neg,
                    )

                m1 = cutmix_box1.unsqueeze(1).expand_as(image_u_s1) == 1
                m2 = cutmix_box2.unsqueeze(1).expand_as(image_u_s2) == 1
                image_u_s1 = image_u_s1.clone(); image_u_s1[m1] = image_u_s1_mix[m1]
                image_u_s2 = image_u_s2.clone(); image_u_s2[m2] = image_u_s2_mix[m2]

                pred_u_s_outs, _ = model(torch.cat((image_u_s1, image_u_s2)))
                pred_u_s1, pred_u_s2 = pred_u_s_outs.chunk(2)

                pred_u_s1_mix, pred_class_u_s1_mix = model(image_u_s1_mix)
                pred_u_s2_mix, pred_class_u_s2_mix = model(image_u_s2_mix)

                (_, _), (pred_class_u_w_mix, pred_class_u_w_mix_fp) = model(image_u_w_mix, True)

                with torch.no_grad():
                    pred_u_w_tea, _ = model(image_u_w)
                    if use_hard_view_mask:
                        pred_u_w_tea = apply_view_mask_logits(pred_u_w_tea, view_u, allowed_seg_mat)
                    conf_u_w = pred_u_w_tea.float().softmax(dim=1).max(dim=1)[0]
                    mask_u_w = pred_u_w_tea.argmax(dim=1)

                mask_u_w_c1 = mask_u_w.clone(); conf_u_w_c1 = conf_u_w.clone()
                mask_u_w_c2 = mask_u_w.clone(); conf_u_w_c2 = conf_u_w.clone()
                mask_u_w_c1[cutmix_box1 == 1] = mask_u_w_mix[cutmix_box1 == 1]
                conf_u_w_c1[cutmix_box1 == 1] = conf_u_w_mix[cutmix_box1 == 1]
                mask_u_w_c2[cutmix_box2 == 1] = mask_u_w_mix[cutmix_box2 == 1]
                conf_u_w_c2[cutmix_box2 == 1] = conf_u_w_mix[cutmix_box2 == 1]

                if use_hard_view_mask:
                    pred_u_w_fp   = apply_view_mask_logits(pred_u_w_fp,   view_u,     allowed_seg_mat)
                    pred_u_s1_mix = apply_view_mask_logits(pred_u_s1_mix, view_u_mix, allowed_seg_mat)
                    pred_u_s2_mix = apply_view_mask_logits(pred_u_s2_mix, view_u_mix, allowed_seg_mat)

                loss_u_s1_seg     = criterion_seg_dice(pred_u_s1.softmax(dim=1),     mask_u_w_c1.unsqueeze(1).float(),  ignore=(conf_u_w_c1 < cfg.conf_thresh).float())
                loss_u_s2_seg     = criterion_seg_dice(pred_u_s2.softmax(dim=1),     mask_u_w_c2.unsqueeze(1).float(),  ignore=(conf_u_w_c2 < cfg.conf_thresh).float())
                loss_u_w_fp_seg   = criterion_seg_dice(pred_u_w_fp.softmax(dim=1),   mask_u_w.unsqueeze(1).float(),     ignore=(conf_u_w    < cfg.conf_thresh).float())
                loss_u_s1_mix_seg = criterion_seg_dice(pred_u_s1_mix.softmax(dim=1), mask_u_w_mix.unsqueeze(1).float(), ignore=(conf_u_w_mix < cfg.conf_thresh).float())
                loss_u_s2_mix_seg = criterion_seg_dice(pred_u_s2_mix.softmax(dim=1), mask_u_w_mix.unsqueeze(1).float(), ignore=(conf_u_w_mix < cfg.conf_thresh).float())

                loss_u_pseudo_cls = 0.5 * (
                    masked_focal_bce_with_logits(pred_class_u_s1_mix, p_t, mask_u_cls, pos_weight, gamma=2.0) +
                    masked_focal_bce_with_logits(pred_class_u_s2_mix, p_t, mask_u_cls, pos_weight, gamma=2.0)
                )
                loss_u_w_mix_cls    = masked_mse(torch.sigmoid(pred_class_u_w_mix),    p_t, mask_u_cls)
                loss_u_w_mix_fp_cls = masked_mse(torch.sigmoid(pred_class_u_w_mix_fp), p_t, mask_u_cls)

                loss += (
                    loss_w["u_s1_seg"]         * loss_u_s1_seg
                    + loss_w["u_s2_seg"]       * loss_u_s2_seg
                    + loss_w["u_w_fp_seg"]     * loss_u_w_fp_seg
                    + loss_w["u_s1_mix_seg"]   * loss_u_s1_mix_seg
                    + loss_w["u_s2_mix_seg"]   * loss_u_s2_mix_seg
                    + loss_w["u_pseudo_cls"]   * loss_u_pseudo_cls
                    + loss_w["u_w_mix_cls"]    * loss_u_w_mix_cls
                    + loss_w["u_w_mix_fp_cls"] * loss_u_w_mix_fp_cls
                )

                seg_mask_ratio = (conf_u_w >= cfg.conf_thresh).float().mean().item()
                cls_mask_ratio = (mask_u_cls > 0).float().mean().item()

                diag_n_unsup_iters += 1
                with torch.no_grad():
                    for c in range(cfg.seg_num_classes):
                        c_mask = (mask_u_w == c)
                        n_pix  = c_mask.float().sum()
                        diag_seg_pixels_total[c] += n_pix
                        if n_pix > 0:
                            c_conf = conf_u_w[c_mask]
                            diag_seg_pixels_accepted[c] += (c_conf >= cfg.conf_thresh).float().sum()
                            diag_seg_conf_sum[c]        += c_conf.sum()

                    for v in range(cfg.view_num_classes):
                        v_mask = (view_u_mix == v)
                        diag_cls_total_per_view[v] += v_mask.float().sum()
                        if v_mask.any():
                            diag_cls_accepted_per_view[v] += (mask_u_cls[v_mask].sum(dim=1) > 0).float().sum()

                    diag_cls_prob_sum   += p_t.sum(dim=0)
                    diag_cls_prob_count += p_t.shape[0]

        #  BACKWARD
        optimizer.zero_grad(set_to_none=True)
        if use_amp and scaler is not None and amp_dtype == torch.float16:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        global_step += 1

        # Meters
        for k, v in [
            ("loss", loss), ("x_seg", loss_x_seg), ("x_cls", loss_x_cls),
            ("x_fp_cls", loss_x_fp_cls), ("u_s1_seg", loss_u_s1_seg),
            ("u_s2_seg", loss_u_s2_seg), ("u_w_fp_seg", loss_u_w_fp_seg),
            ("u_s1_mix_seg", loss_u_s1_mix_seg), ("u_s2_mix_seg", loss_u_s2_mix_seg),
            ("u_pseudo_cls", loss_u_pseudo_cls), ("u_w_mix_cls", loss_u_w_mix_cls),
            ("u_w_mix_fp_cls", loss_u_w_mix_fp_cls),
            ("seg_mask_ratio", seg_mask_ratio), ("cls_mask_ratio", cls_mask_ratio),
        ]:
            meters[k].update(float(v) if not isinstance(v, float) else v)

        # Per-iteration log
        if iters_per_epoch >= 8 and i % max(1, iters_per_epoch // 8) == 0:
            m = meters
            phase = "WARMUP" if is_warmup else "SEMI"
            logger.info(
                f"[{phase}] Iter {i}/{iters_per_epoch} | L={m['loss'].avg:.3f} | "
                f"X(seg={m['x_seg'].avg:.3f} cls={m['x_cls'].avg:.3f} fp={m['x_fp_cls'].avg:.3f}) | "
                f"U(s1={m['u_s1_seg'].avg:.3f} s2={m['u_s2_seg'].avg:.3f} "
                f"fp={m['u_w_fp_seg'].avg:.3f} mix1={m['u_s1_mix_seg'].avg:.3f} "
                f"mix2={m['u_s2_mix_seg'].avg:.3f} pcls={m['u_pseudo_cls'].avg:.3f} "
                f"wcls={m['u_w_mix_cls'].avg:.3f} wfp={m['u_w_mix_fp_cls'].avg:.3f}) | "
                f"mask(seg={m['seg_mask_ratio'].avg:.3f} cls={m['cls_mask_ratio'].avg:.3f})"
            )

        # TensorBoard
        if global_step % 20 == 0:
            writer.add_scalar("train/loss",           meters["loss"].val, global_step)
            writer.add_scalar("train/seg_mask_ratio", seg_mask_ratio,     global_step)
            writer.add_scalar("train/cls_mask_ratio", cls_mask_ratio,     global_step)
            for gi, g in enumerate(optimizer.param_groups):
                writer.add_scalar(f"train/lr_group{gi}", g["lr"], global_step)

        # LR schedule
        iters = epoch * iters_per_epoch + i
        for param_group, blr in zip(optimizer.param_groups, base_lrs):
            param_group["lr"] = poly_lr(blr, iters, total_iters)

    if diag_n_unsup_iters > 0:
        logger.info("=" * 70)
        logger.info(f"PSEUDO-LABEL DIAGNOSTICS — Epoch {epoch}")
        logger.info("=" * 70)

        seg_total = diag_seg_pixels_total.cpu().numpy()
        seg_acc   = diag_seg_pixels_accepted.cpu().numpy()
        seg_conf  = diag_seg_conf_sum.cpu().numpy()

        logger.info("  [SEG] Pseudo-label acceptance per class:")
        for c in range(cfg.seg_num_classes):
            name = SEG_CLASS_NAMES[c] if c < len(SEG_CLASS_NAMES) else f"c{c}"
            if seg_total[c] > 0:
                pct      = 100.0 * seg_acc[c] / seg_total[c]
                avg_conf = seg_conf[c] / seg_total[c]
                status   = " STARVING" if pct < 10.0 else ("" if pct > 50.0 else "~")
                logger.info(f"    {name:>8s}: {pct:5.1f}% accepted | avg_conf={avg_conf:.3f} | {int(seg_total[c]):>8d} px  {status}")
            else:
                logger.info(f"    {name:>8s}: — (no predicho)")

        cls_tv = diag_cls_total_per_view.cpu().numpy()
        cls_av = diag_cls_accepted_per_view.cpu().numpy()
        logger.info("  [CLS] Acceptance per view:")
        for v in range(cfg.view_num_classes):
            name = VIEW_NAMES[v] if v < len(VIEW_NAMES) else f"v{v}"
            if cls_tv[v] > 0:
                pct = 100.0 * cls_av[v] / cls_tv[v]
                logger.info(f"    {name:>4s}: {pct:5.1f}% con ≥1 clase aceptada ({int(cls_av[v])}/{int(cls_tv[v])})")
            else:
                logger.info(f"    {name:>4s}: — (sin muestras)")

        cls_ps = diag_cls_prob_sum.cpu().numpy()
        cls_pc = diag_cls_prob_count.cpu().numpy()
        logger.info("  [CLS] Probabilidad promedio del teacher por enfermedad:")
        for k in range(cfg.cls_num_classes):
            name = CLS_CLASS_NAMES[k] if k < len(CLS_CLASS_NAMES) else f"d{k}"
            if cls_pc[k] > 0:
                logger.info(f"    {name:>10s}: avg_prob={cls_ps[k]/cls_pc[k]:.4f}")

        logger.info("=" * 70)
    else:
        logger.info(f"Epoch {epoch}: WARMUP - no pseudo-label diagnostics")

    return meters, global_step

SEG_CLASS_NAMES = ["BG","LA","RA","LV","RV","WH","TV","DA","Ao","PA","PV","RVout","SVC","AoArch","Duct"]
CLS_CLASS_NAMES = ["VSD", "AV_sten", "AoHypo", "AoV_sten", "DORV", "PV_sten", "RAA"]
VIEW_NAMES      = ["4CH","LVOT","RVOT","3VT"]

print("train_one_epoch definido ")


In [ ]:
os.makedirs(cfg.checkpoint_dir, exist_ok=True)
os.makedirs(cfg.log_dir,        exist_ok=True)
os.makedirs(cfg.result_dir,     exist_ok=True)

logger = setup_logger(cfg.log_dir)
logger.info(f"version={cfg.version} | epochs={cfg.epochs}")

writer = SummaryWriter(log_dir=cfg.log_dir)
print(f"TensorBoard  {cfg.log_dir}")


In [ ]:
@torch.no_grad()
def validate(model, valid_loader, allowed_seg_mat, cls_allowed, cfg, logger, epoch, best_score):
    model.eval()
    C, K, tol = cfg.seg_num_classes, cfg.cls_num_classes, 2.0

    dice_sum = np.zeros(C - 1, dtype=np.float64)
    nsd_sum  = np.zeros(C - 1, dtype=np.float64)
    cnt      = np.zeros(C - 1, dtype=np.int64)
    y_true_all, y_prob_all, views_all = [], [], []

    for image, view, mask, class_label in valid_loader:
        image       = image.to(device)
        gt_mask     = mask.to(device)
        class_label = class_label.to(device)
        v           = view.to(device).long().view(-1)

        h, w = image.shape[-2:]
        image_rs = F.interpolate(image, (cfg.resize_target, cfg.resize_target),
                                 mode="bilinear", align_corners=False)

        seg_logits, cls_out = model(image_rs)

        logits = F.interpolate(seg_logits, (h, w), mode="bilinear", align_corners=False)
        logits = apply_view_mask_logits(logits, v, allowed_seg_mat)
        probs = logits.softmax(dim=1)

        pm = probs.argmax(dim=1).cpu().numpy()[0].astype(np.int32)
        gt = gt_mask.cpu().numpy()[0].astype(np.int32)

        for cls in range(1, C):
            pred_bin = (pm == cls); gt_bin = (gt == cls)
            union = pred_bin.sum() + gt_bin.sum()
            if union == 0: continue
            inter = (pred_bin & gt_bin).sum()
            dice_sum[cls-1] += (2.0 * inter) / (union + 1e-8)
            nsd_sum[cls-1]  += nsd_binary(pred_bin, gt_bin, tol=tol)
            cnt[cls-1]      += 1

        prob = torch.sigmoid(cls_out)
        y_true_all.append(class_label.cpu().numpy()[0])
        y_prob_all.append(prob.cpu().numpy()[0])
        views_all.append(v.cpu().numpy()[0])

    dice_class = 100.0 * dice_sum / np.maximum(cnt, 1)
    nsd_class  = 100.0 * nsd_sum  / np.maximum(cnt, 1)
    valid_mask = cnt > 0
    mean_dice  = float(dice_class[valid_mask].mean()) if valid_mask.any() else 0.0
    mean_nsd   = float(nsd_class[valid_mask].mean())  if valid_mask.any() else 0.0

    y_true_all = np.stack(y_true_all) if y_true_all else np.zeros((0, K))
    y_prob_all = np.nan_to_num(
        np.stack(y_prob_all) if y_prob_all else np.zeros((0, K)), nan=0.0)
    views_all  = np.array(views_all, dtype=np.int32)

    metrics  = masked_metrics_with_threshold_search(y_true_all, y_prob_all, views_all, cls_allowed)
    macro_f1 = float(metrics["macro_f1@0.5"])
    score    = (mean_dice + mean_nsd) / 2.0 + macro_f1 * 100.0

    logger.info(f"[Ep {epoch}] Dice={mean_dice:.2f} NSD={mean_nsd:.2f} "
                f"F1@0.5={macro_f1:.4f} F1@best={metrics['macro_f1@best']:.4f} "
                f"Score={score:.4f} | Best={best_score:.4f}")
    for cls_idx in range(C - 1):
        logger.info(f"  Seg Class[{cls_idx+1:02d}] Dice={dice_class[cls_idx]:.1f} "
                    f"NSD={nsd_class[cls_idx]:.1f} N={cnt[cls_idx]}")
    for k in range(K):
        logger.info(f"  Cls[{k}] F1={metrics['per_class_f1@0.5'][k]:.4f} "
                    f"AUPRC={metrics['per_class_auprc'][k]:.4f} "
                    f"support={metrics['support'][k]}")

    return {
        "dice_class": dice_class, "nsd_class": nsd_class, "cnt": cnt,
        "mean_dice": mean_dice, "mean_nsd": mean_nsd,
        "metrics": metrics, "macro_f1": macro_f1, "score": float(score),
    }

print("validate defined (no TTA)")

In [ ]:

from src.utils import load_pretrained_flexible

def maybe_resume(model, optimizer, scaler, ckpt_path, logger):
    """Load from latest.pth if it exists. Returns (start_epoch, best_score, best_epoch, global_step)."""
    if not os.path.exists(ckpt_path):
        logger.info("[Resume] No checkpoint found, training from scratch.")
        return -1, 0.0, -1, 0

    ckpt = load_pretrained_flexible(model, ckpt_path, logger=logger, key="model")
    start_epoch = int(ckpt.get("epoch", -1))
    best_score  = float(ckpt.get("previous_best", 0.0))
    best_epoch  = int(ckpt.get("best_epoch", -1))
    global_step = int(ckpt.get("global_step", 0))

    if "optimizer" in ckpt:
        try:
            optimizer.load_state_dict(ckpt["optimizer"])
            logger.info("[Resume] Optimizer state loaded.")
        except ValueError as e:
            logger.warning(f"[Resume] Skip optimizer (incompatible): {e}")

    if scaler is not None and ckpt.get("scaler") is not None:
        try:
            scaler.load_state_dict(ckpt["scaler"])
            logger.info("[Resume] GradScaler loaded.")
        except Exception as e:
            logger.warning(f"[Resume] Skip GradScaler: {e}")

    logger.info(f"[Resume] epoch={start_epoch}, best={best_score:.4f}@ep{best_epoch}, step={global_step}")
    return start_epoch, best_score, best_epoch, global_step



os.makedirs(cfg.checkpoint_dir, exist_ok=True)
os.makedirs(cfg.log_dir,        exist_ok=True)
os.makedirs(cfg.result_dir,     exist_ok=True)

logger = setup_logger(cfg.log_dir)
writer = SummaryWriter(log_dir=cfg.log_dir)

logger.info(f"version={cfg.version} | epochs={cfg.epochs}")
logger.info(f"SEG_ALLOWED: {SEG_ALLOWED}")
logger.info(f"CLS_ALLOWED: {CLS_ALLOWED}")
logger.info(f"LOSS_WEIGHTS: {LOSS_WEIGHTS}")
logger.info(f"pseudo_tau_pos={cfg.pseudo_tau_pos}, pseudo_tau_neg={cfg.pseudo_tau_neg}")
logger.info(f"TensorBoard  {cfg.log_dir}")


total_iters = len(train_loader_u) * cfg.epochs

ckpt_path = os.path.join(cfg.checkpoint_dir, "latest.pth")
start_epoch, best_score, best_epoch, global_step = maybe_resume(
    model, optimizer, scaler, ckpt_path, logger
)

history = {"train_loss": [], "val_dice": [], "val_nsd": [], "val_f1": [], "val_score": []}

logger.info(f"Start | iters/epoch={len(train_loader_u)} | total_iters={total_iters}")
logger.info("=" * 60)


for epoch in range(start_epoch + 1, cfg.epochs):
    logger.info(
        f"==> Epoch {epoch}/{cfg.epochs} | LR={optimizer.param_groups[0]['lr']:.6f} "
        f"| Best={best_score:.4f}@ep{best_epoch}"
    )

    meters, global_step = train_one_epoch(
        model, optimizer, scaler,
        train_loader_l, train_loader_u, train_loader_u_mix,
        allowed_seg_mat, allowed_cls_mat, pos_weight,
        criterion_seg_ce, criterion_seg_dice,

        LOSS_WEIGHTS,

        base_lrs,

        epoch, total_iters,
        cfg, writer, global_step, logger
    )
    history["train_loss"].append(meters["loss"].avg)


    val = validate(
        model, valid_loader, allowed_seg_mat, CLS_ALLOWED,
        cfg, logger, epoch, best_score
    )
    history["val_dice"].append(val["mean_dice"])
    history["val_nsd"].append(val["mean_nsd"])
    history["val_f1"].append(val["macro_f1"])
    history["val_score"].append(val["score"])

    writer.add_scalar("val/mean_dice", val["mean_dice"], epoch)
    writer.add_scalar("val/mean_nsd",  val["mean_nsd"],  epoch)
    writer.add_scalar("val/macro_f1",  val["macro_f1"],  epoch)
    writer.add_scalar("val/score",     val["score"],     epoch)

    score = val["score"]
    is_best = score > best_score
    if is_best:
        best_score = score
        best_epoch = epoch

    logger.info(
        f"[Val] Epoch {epoch:03d} | score={score:.4f} | "
        f"best={best_score:.4f}@ep{best_epoch:03d} | is_best={is_best}"
    )


    checkpoint = {
        "model":         model.state_dict(),
        "optimizer":     optimizer.state_dict(),
        "scaler":        scaler.state_dict() if scaler is not None else None,
        "epoch":         epoch,
        "previous_best": best_score,
        "best_epoch":    best_epoch,
        "global_step":   global_step,
    }

    # latest.pth EVERY epoch (matching .py line 837)
    torch.save(checkpoint, os.path.join(cfg.checkpoint_dir, "latest.pth"))

    if is_best:
        torch.save(checkpoint, os.path.join(cfg.checkpoint_dir, "best.pth"))
        logger.info(f"   New best  {cfg.checkpoint_dir}/best.pth")

logger.info("=" * 60)
logger.info(f"DONE | best={best_score:.4f} @ ep{best_epoch}")
writer.close()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

val_freq = getattr(cfg, "val_freq", 1)  # default: validate every epoch
val_epochs = list(range(0, len(history["val_dice"]) * val_freq, val_freq))

axes[0,0].plot(history["train_loss"]); axes[0,0].set_title("Train Loss"); axes[0,0].grid(alpha=0.3)
axes[0,1].plot(val_epochs, history["val_dice"], marker='o'); axes[0,1].set_title("Val Dice (%)"); axes[0,1].grid(alpha=0.3)
axes[1,0].plot(val_epochs, history["val_nsd"],  marker='o', color='green'); axes[1,0].set_title("Val NSD (%)"); axes[1,0].grid(alpha=0.3)
axes[1,1].plot(val_epochs, history["val_f1"],   marker='o', color='red'); axes[1,1].set_title("Val Macro F1"); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(cfg.result_dir, "training_history.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
history_file = os.path.join(cfg.result_dir, "training_history.json")
with open(history_file, "w") as f:
    json.dump({k: [float(x) for x in v] for k, v in history.items()}, f, indent=2)
print(f"Saved: {history_file}")
print(f"Best score: {best_score:.4f} @ epoch {best_epoch}")
